# Deezer APi test

In this notebook, I'll try to do the following tasks 

1 -> play with the deezer api, like ger comfortable with it

2 -> check the preview songs and if they can be downloaded

3 -> create small dataset to work with

In [1]:
import requests

url = 'https://api.deezer.com/search'
params = {'q': 'artist track'}
data = requests.get(url, params=params).json()

preview_url = data['data'][0]['preview']

audio = requests.get(preview_url).content
with open('preview.mp3', 'wb') as f:
    f.write(audio)

## Reusable Deezer helpers

The next cells let you:
- search songs by text query
- build a clean songs table (DataFrame)
- download 30s preview clips in batch

Note: Deezer provides preview snippets, not full songs.

In [2]:
import os
import re
from pathlib import Path

import pandas as pd
import requests

DEEZER_SEARCH_URL = "https://api.deezer.com/search"


def search_tracks(query: str, limit: int = 25, timeout: int = 20) -> list[dict]:
    """Search tracks on Deezer and return raw track dictionaries."""
    params = {"q": query, "limit": limit}
    r = requests.get(DEEZER_SEARCH_URL, params=params, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    return data.get("data", [])


def tracks_to_df(tracks: list[dict]) -> pd.DataFrame:
    """Convert Deezer track dictionaries into a tidy DataFrame."""
    rows = []
    for t in tracks:
        artist = t.get("artist", {}) or {}
        album = t.get("album", {}) or {}
        rows.append(
            {
                "id": t.get("id"),
                "title": t.get("title"),
                "title_short": t.get("title_short"),
                "artist": artist.get("name"),
                "album": album.get("title"),
                "duration_s": t.get("duration"),
                "rank": t.get("rank"),
                "preview_url": t.get("preview"),
                "deezer_link": t.get("link"),
            }
        )
    return pd.DataFrame(rows)


def safe_filename(name: str) -> str:
    cleaned = re.sub(r"[^a-zA-Z0-9._ -]", "", name).strip()
    return cleaned.replace(" ", "_")


def download_previews(df: pd.DataFrame, out_dir: str = "previews", max_files: int | None = None) -> pd.DataFrame:
    """Download preview mp3 files listed in DataFrame[preview_url]."""
    out_path = Path(out_dir)
    out_path.mkdir(parents=True, exist_ok=True)

    subset = df.copy()
    subset = subset[subset["preview_url"].notna()]
    if max_files is not None:
        subset = subset.head(max_files)

    file_paths = []
    for _, row in subset.iterrows():
        title = row.get("title") or "unknown_title"
        artist = row.get("artist") or "unknown_artist"
        filename = safe_filename(f"{artist}-{title}.mp3")
        path = out_path / filename

        audio = requests.get(row["preview_url"], timeout=20).content
        with open(path, "wb") as f:
            f.write(audio)
        file_paths.append(str(path))

    subset = subset.copy()
    subset["local_preview_path"] = file_paths
    return subset.reset_index(drop=True)

In [4]:
# Reliable workflow: get top tracks from a specific artist

def find_artist_id(artist_name: str, timeout: int = 20) -> int:
    r = requests.get("https://api.deezer.com/search/artist", params={"q": artist_name}, timeout=timeout)
    r.raise_for_status()
    artists = r.json().get("data", [])
    if not artists:
        raise ValueError(f"No artist found for: {artist_name}")

    # Prefer exact name match if available
    exact = [a for a in artists if a.get("name", "").lower() == artist_name.lower()]
    return (exact[0] if exact else artists[0])["id"]


def get_artist_top_tracks(artist_name: str, limit: int = 15, timeout: int = 20) -> pd.DataFrame:
    artist_id = find_artist_id(artist_name, timeout=timeout)
    r = requests.get(f"https://api.deezer.com/artist/{artist_id}/top", params={"limit": limit}, timeout=timeout)
    r.raise_for_status()
    tracks = r.json().get("data", [])
    return tracks_to_df(tracks)


artist_name = "Daft Punk"
df = get_artist_top_tracks(artist_name=artist_name, limit=15)

print(f"Found {len(df)} top tracks for {artist_name}")
display(df[["artist", "title", "album", "preview_url"]].head(10))

# Download up to 5 preview clips
preview_df = download_previews(df, out_dir="previews_daftpunk", max_files=5)
display(preview_df[["artist", "title", "local_preview_path"]])

Found 15 top tracks for Daft Punk


,artist,title,album,preview_url
0,Daft Punk,One More Time,Discovery,https://cdnt-preview.dzcdn.net/api/1/1/f/8/c/0...
1,Daft Punk,Get Lucky (Radio Edit - feat. Pharrell William...,Get Lucky (Radio Edit - feat. Pharrell William...,https://cdnt-preview.dzcdn.net/api/1/1/1/b/f/0...
2,Daft Punk,Get Lucky (feat. Pharrell Williams and Nile Ro...,Random Access Memories (10th Anniversary Edition),https://cdnt-preview.dzcdn.net/api/1/1/c/8/a/0...
3,Daft Punk,Technologic,Human After All,https://cdnt-preview.dzcdn.net/api/1/1/a/e/f/0...
4,Daft Punk,Instant Crush (feat. Julian Casablancas),Random Access Memories,https://cdnt-preview.dzcdn.net/api/1/1/d/6/b/0...
5,Daft Punk,Veridis Quo,Discovery,https://cdnt-preview.dzcdn.net/api/1/1/6/4/5/0...
6,Daft Punk,"Harder, Better, Faster, Stronger",Discovery,https://cdnt-preview.dzcdn.net/api/1/1/6/a/2/0...
7,Daft Punk,Around the World,Homework,https://cdnt-preview.dzcdn.net/api/1/1/a/4/7/0...
8,Daft Punk,Da Funk,Homework,https://cdnt-preview.dzcdn.net/api/1/1/7/6/8/0...
9,Daft Punk,Give Life Back to Music,Random Access Memories,https://cdnt-preview.dzcdn.net/api/1/1/f/5/7/0...


,artist,title,local_preview_path
0,Daft Punk,One More Time,previews_daftpunk/Daft_Punk-One_More_Time.mp3
1,Daft Punk,Get Lucky (Radio Edit - feat. Pharrell William...,previews_daftpunk/Daft_Punk-Get_Lucky_Radio_Ed...
2,Daft Punk,Get Lucky (feat. Pharrell Williams and Nile Ro...,previews_daftpunk/Daft_Punk-Get_Lucky_feat._Ph...
3,Daft Punk,Technologic,previews_daftpunk/Daft_Punk-Technologic.mp3
4,Daft Punk,Instant Crush (feat. Julian Casablancas),previews_daftpunk/Daft_Punk-Instant_Crush_feat...


In [5]:
from pathlib import Path
import os

cwd = Path(os.getcwd())
tracks_dir = cwd / "previews_daftpunk"
print("Notebook cwd:", cwd)
print("Tracks directory:", tracks_dir)
print("Exists:", tracks_dir.exists())
if tracks_dir.exists():
    for p in sorted(tracks_dir.glob("*.mp3")):
        print(p.resolve())

Notebook cwd: /home/filipepimentel
Tracks directory: /home/filipepimentel/previews_daftpunk
Exists: True
/home/filipepimentel/previews_daftpunk/Benny_Benassi-Satisfaction_Isak_Original_Extended.mp3
/home/filipepimentel/previews_daftpunk/Daft_Punk-Get_Lucky_Radio_Edit_-_feat._Pharrell_Williams_and_Nile_Rodgers.mp3
/home/filipepimentel/previews_daftpunk/Daft_Punk-Get_Lucky_feat._Pharrell_Williams_and_Nile_Rodgers.mp3
/home/filipepimentel/previews_daftpunk/Daft_Punk-Instant_Crush_feat._Julian_Casablancas.mp3
/home/filipepimentel/previews_daftpunk/Daft_Punk-One_More_Time.mp3
/home/filipepimentel/previews_daftpunk/Daft_Punk-Technologic.mp3
/home/filipepimentel/previews_daftpunk/Manny_Da_Prince-Beast.mp3
/home/filipepimentel/previews_daftpunk/Pan_Da_Punk-Bianco_e_Nero.mp3
/home/filipepimentel/previews_daftpunk/Pan_Da_Punk-Breaking_Italy.mp3
/home/filipepimentel/previews_daftpunk/Pan_Da_Punk-Non_Ci_Sono.mp3
